# Model initialization|

In [1]:
from transformers import RTDetrImageProcessor, RTDetrV2ForObjectDetection

image_processor = RTDetrImageProcessor.from_pretrained("PekingU/rtdetr_v2_r101vd")
model = RTDetrV2ForObjectDetection.from_pretrained("PekingU/rtdetr_v2_r101vd", device_map="auto")

Loading weights:   0%|          | 0/1025 [00:00<?, ?it/s]

# Loading images

In [2]:
from glob import glob

img_paths = glob(r'D:\AIC_LITE\kfs\Keyframes_L01\keyframes\*\*.jpg')
img_codes = [f'{path.split('\\')[-2]}_{path.split('\\')[-1].split('.')[0]}' for path in img_paths]

# Detect objects

In [3]:
import torch
from tqdm import tqdm
from PIL import Image

obj_dets = {}

BATCH_SIZE = 16
CONFIDENCE_THRESH = 0.7
for i in tqdm(range(0, len(img_paths), BATCH_SIZE)):
    j = len(img_paths) if i + BATCH_SIZE > len(img_paths) else i + BATCH_SIZE
    paths = img_paths[i:j]
    codes = img_codes[i:j]

    raws = [Image.open(path) for path in paths]
    inputs = image_processor(images=raws, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)

    results = image_processor.post_process_object_detection(
        outputs,
        target_sizes=torch.tensor([(raw.height, raw.width) for raw in raws]),
        threshold=CONFIDENCE_THRESH
    )
    for code, result in zip(codes, results):
        det = []
        for score, label_id, box in zip(result["scores"], result["labels"], result["boxes"]):
            score, label = score.item(), model.config.id2label[label_id.item()]
            box = [round(i, 2) for i in box.tolist()]
            det.append({'label': label, 'score': score, 'box': box})
        obj_dets[code] = det

100%|██████████| 508/508 [06:51<00:00,  1.24it/s]


# Saving detections

In [4]:
import json
from tqdm import tqdm

for code, det in tqdm(list(obj_dets.items())):
    parent = r'..\\additional_data\\obj_det'
    with open(f'{parent}\\{code}.json', 'w') as f:
        json.dump(det, f)

100%|██████████| 8115/8115 [00:05<00:00, 1572.44it/s]
